In [ ]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "5"

In [ ]:
import scanpy as sc
import scvi 
import plotnine as gg
import numpy as np
import matplotlib.pyplot as plt

from essential.utils import PLOTNINE_DEFAULT_THEME_2

In [ ]:
adata_path1 = "/workspace/data/Nov2025_DE122_genomescale_EZRDM_Glu_newpipeline_preprocessed/Nov2025_DE122_genomescale_EZRDM_Glu_newpipeline_preprocessed.h5ad"
adata_path2 = "/workspace/data/260309_lce75_genomescale_ezrdm_glu_preprocessed.h5ad"
adata1 = sc.read_h5ad(adata_path1)
adata1.obs["experiment"] = "de122"
adata2 = sc.read_h5ad(adata_path2)
adata2.obs["experiment"] = "lce75"

adata = sc.concat([adata1, adata2])
adata_ = adata.copy()
TARGET_KEY = "top_target_unthresholded"
adata_ = adata_[~adata_.obs[TARGET_KEY].isna()].copy()
adata_

In [ ]:
# DE122 more stringent on UMIs
# LCE75 less stringent (poorer genotyping)

In [ ]:
adata1.obs["guide_n_umis"].describe()

In [ ]:
adata.obs["guide_n_umis"]

In [ ]:
adata.obs["guide_purity_all_reads"]

In [ ]:
# fraction of UMIs that map to the called guide
adata.obs["guide_purity"]

# target is not NA based on 
# of UMIs: something like >= 3
# and based on purity

In [ ]:
adata_.obs["top_target_unthresholded"]

In [ ]:
adata_

In [ ]:
scvi.model.SCVI.setup_anndata(adata_, categorical_covariate_keys=["rt_bc", "experiment"], layer="reads")
model = scvi.model.SCVI(adata_)
model.train()

In [ ]:
adata

In [ ]:
latent = model.get_latent_representation()
adata_.obsm["latent"] = latent

In [ ]:
sc.pp.neighbors(adata_, use_rep="latent")
sc.tl.umap(adata_)

adata_.obs["UMAP1"] = adata_.obsm["X_umap"][:, 0]
adata_.obs["UMAP2"] = adata_.obsm["X_umap"][:, 1]

(
    gg.ggplot(adata_.obs, gg.aes(x="UMAP1", y="UMAP2", color="rt_bc"))
    + gg.geom_point(size=0.5, stroke=0.0)
    + gg.theme_classic()
    + PLOTNINE_DEFAULT_THEME_2
    + gg.labs(
        title="scVI latent space (batch correction)"
    )
)


In [ ]:
(
    gg.ggplot(adata_.obs, gg.aes(x="UMAP1", y="UMAP2"))
    + gg.geom_point(size=0.5, stroke=0.0, color="grey")
    + gg.geom_point(adata_.obs.loc[lambda x: x[TARGET_KEY] == 'nontargeting']), size=2, stroke=0.0, color="red")
    + gg.theme_classic()
    + PLOTNINE_DEFAULT_THEME_2
)

In [ ]:
sc.tl.leiden(adata_, resolution=0.4)

In [ ]:
adata_.obs["is_control"] = adata_.obs["leiden"].isin(["0", "1", "2", "3", "4", "5"])

In [ ]:
(
    gg.ggplot(adata_.obs, gg.aes(x="UMAP1", y="UMAP2", color="leiden"))
    + gg.geom_point()
    + gg.theme_classic()
    # + PLOTNINE_DEFAULT_THEME_2
    + gg.theme(
        # legend_position="none"
    )
    + gg.labs(
        title="scVI latent space (batch correction)"
    )
    
)

In [ ]:
adata_subset = adata_[adata_.obs[TARGET_KEY].isin(["lpxB", "lpxK", "lpxC", "lpdxD"])].copy()
adata_subset.obs[TARGET_KEY] = adata_subset.obs[TARGET_KEY].astype(str)

(
    gg.ggplot(adata_.obs, gg.aes(x="UMAP1", y="UMAP2"))
    + gg.geom_point()
    + gg.geom_point(gg.aes(color=TARGET_KEY), adata_subset.obs, size=5.0, stroke=0.0)
    + gg.theme_classic()
    # + PLOTNINE_DEFAULT_THEME_2
    + gg.theme(
        # legend_position="none"
    )
    + gg.labs(
        title="scVI latent space (batch correction)"
    )
    
)

In [ ]:
(
    gg.ggplot(adata_.obs, gg.aes(x="UMAP1", y="UMAP2", color="is_control"))
    + gg.geom_point()
    + gg.theme_classic()
    # + PLOTNINE_DEFAULT_THEME_2
    + gg.theme(
        # legend_position="none"
    )
    + gg.labs(
        title="scVI latent space (batch correction)"
    )
    
)

In [ ]:
(adata_case.obs["target"] == "nontargeting").sum()

In [ ]:
adata_case = adata_[~adata_.obs["is_control"]].copy()
adata_case

In [ ]:
sc.pp.neighbors(adata_case, use_rep="latent", n_neighbors=15)
sc.tl.umap(adata_case, min_dist=0.5)
sc.tl.leiden(adata_case, resolution=1.0)

adata_case.obs["WITH_PHENOTYPE_UMAP1"] = adata_case.obsm["X_umap"][:, 0]
adata_case.obs["WITH_PHENOTYPE_UMAP2"] = adata_case.obsm["X_umap"][:, 1]

(
    gg.ggplot(adata_case.obs, gg.aes(x="WITH_PHENOTYPE_UMAP1", y="WITH_PHENOTYPE_UMAP2", color="rt_bc"))
    + gg.geom_point(size=1.0, stroke=0.0)
    # + gg.theme_classic()
    # + PLOTNINE_DEFAULT_THEME_2
    + gg.labs(
        title="scVI latent space (batch correction)"
    )
)


In [ ]:
adata_subset = adata_case[adata_case.obs["target"].isin(["lpxB", "lpxK", "lpxC", "lpdxD"])].copy()
adata_subset.obs["target"] = adata_subset.obs["target"].astype(str)
(
    gg.ggplot(adata_case.obs, gg.aes(x="WITH_PHENOTYPE_UMAP1", y="WITH_PHENOTYPE_UMAP2"))
    + gg.geom_point(size=1.0, stroke=0.0)
    + gg.geom_point(gg.aes(color="target"), adata_subset.obs, size=5.0, stroke=0.0)
    # + gg.theme_classic()
    # + PLOTNINE_DEFAULT_THEME_2
    + gg.labs(
        title="scVI latent space (batch correction)"
    )
)


In [ ]:
import plotly.express as px
import pandas as pd


fig = px.scatter(adata_case.obs, x="WITH_PHENOTYPE_UMAP1", y="WITH_PHENOTYPE_UMAP2", color="target", hover_data=["target"],)
fig.update_traces(marker=dict(size=3))
fig.update_layout(showlegend=False)
fig.show()

In [ ]:
scvi.model.SCVI.setup_anndata(adata_case, categorical_covariate_keys=["rt_bc", "experiment"], layer="reads")
model_case = scvi.model.SCVI(adata_case)
model_case.train()

In [ ]:
latent_refit = model_case.get_latent_representation()
adata_case.obsm["latent_refit"] = latent_refit

In [ ]:
sc.pp.neighbors(adata_case, use_rep="latent_refit")
sc.tl.umap(adata_case, min_dist=0.2)

adata_case.obs["REFIT_WITH_PHENOTYPE_UMAP1"] = adata_case.obsm["X_umap"][:, 0]
adata_case.obs["REFIT_WITH_PHENOTYPE_UMAP2"] = adata_case.obsm["X_umap"][:, 1]

(
    gg.ggplot(adata_case.obs, gg.aes(x="REFIT_WITH_PHENOTYPE_UMAP1", y="REFIT_WITH_PHENOTYPE_UMAP2"))
    + gg.geom_point(size=0.5, stroke=0.0)
    + gg.theme_classic()
    + PLOTNINE_DEFAULT_THEME_2
    + gg.labs(
        title="scVI latent space (batch correction)"
    )
)


In [ ]:
import plotly.express as px
import pandas as pd


fig = px.scatter(adata_case.obs, x="REFIT_WITH_PHENOTYPE_UMAP1", y="REFIT_WITH_PHENOTYPE_UMAP2", color="target", hover_data=["target"],)
fig.update_traces(marker=dict(size=3))
fig.update_layout(showlegend=False)
fig.show()